In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from qdrant_client.models import PointStruct
import tqdm as notebook_tqdm
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader, Docx2txtLoader


/home/anujkumar/resumeRanking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding_function = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-mpnet-base-v2"
)

In [28]:
# loading resumes
def doc_loader():
    loader = DirectoryLoader(
        path="../cv",
        glob="**/*.pdf",
        loader_cls=PyMuPDFLoader,
        show_progress = True
    )
    docs = loader.load()
    loader = DirectoryLoader(
        path="../cv",
        glob="**/*.docx",
        loader_cls=Docx2txtLoader,
        show_progress=True
    )
    docs.extend(loader.load())
    return docs

In [29]:
docs = doc_loader()
print(f"{len(docs)} number of documents loaded successfully")

100%|██████████| 5/5 [00:00<00:00, 159.51it/s]

12 number of documents loaded successfully


In [32]:
docs[0]

Document(metadata={'producer': 'pdfTeX-1.40.23', 'creator': 'TeX', 'creationdate': '2022-01-04T16:53:23+00:00', 'source': '../cv/AnuvaGoyal_Latex.pdf', 'file_path': '../cv/AnuvaGoyal_Latex.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2022-01-04T16:53:23+00:00', 'trapped': '', 'modDate': 'D:20220104165323Z', 'creationDate': 'D:20220104165323Z', 'page': 0}, page_content='ANUVA GOYAL\n[ anuvagoyal111@gmail.com\n½ Agra, Uttar Pradesh, India\n\x87 github.com/AnuvaGoyal\nPROJECTS\n• Mental Healthcare Chatbot that provides ad-\nvice to the user based on diﬀerent categories\nof mental health problems using a dataset\nwebscraped from counselchat.com (Nov 2021)\n• Full stack Speech Emotion based Movie\nRecommender System using the RAVDESS\nDataset and Web Scraping techniques (Oct\n2021)\n• Finding a Perfect Fit, a model to parse re-\nsumes using Pytesseract, NLP and XG Boost\nand Random Forest classiﬁcation techniques\n(Aug 20

In [41]:
# creating vector of cv_text
qdrant_client = QdrantClient(
    url='localhost:6333'
)
qdrant_client.create_collection(
    collection_name="cv_embeddings",
    vectors_config = VectorParams(size=768, distance=Distance.COSINE)
)
qdrant_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name="cv_embeddings",
    embedding=embedding_function
)

In [42]:
def ensure_cv_collection():
    try:
        qdrant_store = QdrantVectorStore(
            client=qdrant_client,
            collection_name='cv_embeddings',
            embedding=embedding_function
        )
        print(f"Collection is already is present")
    except:
        qdrant_client.create_collection(
            collection_name='cv_embeddings',
            vectors_config=VectorParams(size=768, distance=Distance.COSINE)
        )
        print(f"New Collection Created")

In [43]:
ensure_cv_collection()

Collection is already is present


In [61]:
import hashlib
import uuid
def get_file_hash(file_path):
    with open(file_path, 'rb') as f:
        hex_digest = hashlib.sha256(f.read()).hexdigest()
        return str(uuid.UUID(hex_digest[:32]))

In [62]:
get_file_hash("/home/anujkumar/resumeRanking/cv/1901841_RESUME.pdf")

'329ff176-515b-32bc-d21c-aaf37ec85c62'

In [63]:
"/home/anujkumar/resumeRanking/cv/1901841_RESUME.pdf".endswith(".pdf")

True

In [64]:
def upload_cv(cv_path):
    if not os.path.exists(cv_path):
        return "Path doesn't exists."
    if cv_path.endswith('.pdf'):
        loader=PyMuPDFLoader(cv_path)
    elif cv_path.endswith('.docx') or cv_path.endswith('.doc'):
        loader=Docx2txtLoader(cv_path)
    else:
        return "File Format is not supported. Please provide pdf or docx"
    
    docs = loader.load()
    cv_text = docs[0].page_content
    cv_vec = embedding_function.embed_query(cv_text)

    payload = {
        "total_exp": 0,
        "exp_txt": "Exp section"
    }
    point = PointStruct(
        id=get_file_hash(cv_path),
        vector = cv_vec,
        payload = payload
    )

    qdrant_client.upsert(
        collection_name='cv_embeddings',
        points = [point]
    )
    return f"{cv_path} has been loaded with in the cv_embeddings collection"
    

In [65]:
upload_cv("/home/anujkumar/resumeRanking/cv/react-developer-resume-example.pdf")

'/home/anujkumar/resumeRanking/cv/react-developer-resume-example.pdf has been loaded with in the cv_embeddings collection'

# Testing the qdrant retrival

In [3]:
from qdrant_client.models import Filter, FieldCondition, MatchAny


In [4]:
filter_resume_ids = ['c902d79d-3d54-59db-bf0c-932b1e039aa1']

In [5]:
resume_filter = Filter(
    must = [
        FieldCondition(
            key='id',
            match=MatchAny(any=filter_resume_ids)
        )
    ]
)

In [6]:
from langchain_qdrant import QdrantVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from config import Config
from dotenv import load_dotenv
load_dotenv()
config = Config()
print(config.EMBEDDING_FUNCTION)
qdrant_store = QdrantVectorStore(
    client=config.QDRANT_CLIENT, 
    collection_name=config.CV_COLLECTION, 
    embedding=HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODEL)
)

model_name='sentence-transformers/all-mpnet-base-v2' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [7]:
retriver = qdrant_store.as_retriever(
    search_kwargs={"filter": resume_filter}, k=3
)

In [8]:
retriver.invoke("frontend")

[Document(metadata={'_id': 'c902d79d-3d54-59db-bf0c-932b1e039aa1', '_collection_name': 'cv_embeddings'}, page_content='GINA\nJONES\nReact Developer\ng.jones@email.com\n(123) 456-7890\nSan Francisco, CA\nLinkedIn\nWork Experience\nAirbnb\nReact Developer\n2020 - current | San Francisco, CA\nEnforced code quality across the React codebase with ESLint, resulting\nin a 21% reduction in code review time and improved overall\ncodebase health.\nCreated comprehensive test suites using Enzyme, which increased test\ncoverage by 52%.\nDeveloped server-side applications using NestJS that reduced response\ntime by 44%.\nNetflix\nFront-end Developer\n2017 - 2020 | Los Gatos, CA\nIntegrated Redux for state management, optimizing application\nperformance and reducing load time by 47%.\nUsed Apollo Client for GraphQL integration, which resulted in a 53%\nreduction in API response time.\nAdobe\nFull-stack Developer\n2015 - 2017 | San Jose, CA\nDesigned a responsive user interface using Chakra UI, which 

In [ ]:
Okay, I'm ready to analyze some resumes!  To give you the best possible rating (1-10) for each candidate, I need the actual resumes.  I can't assess their relevance without seeing the content of their work experience, skills, and projects.\n\n**However, I can tell you what I'll be looking for and how I'll weigh different aspects of the resumes based on the job description:**\n\n**High Priority (Critical for a High Rating):**\n\n*   **Node.js Experience:**  This is a core required skill.  I'll be looking for specific projects or roles where the candidate used Node.js extensively.  The more detail, the better.\n*   **Docker Experience:**  Essential for modern deployment.  I'll look for mentions of Docker in their experience, ideally with details about how they used it (e.g., containerizing applications, managing Docker Compose files, working with Docker Swarm or Kubernetes).\n*   **CI/CD Experience:**  The job description explicitly mentions automating deployment pipelines.  I'll be looking for experience with CI/CD tools like Jenkins, GitLab CI, CircleCI, Travis CI, or similar.  Details about building and maintaining CI/CD pipelines are crucial.\n*   **JavaScript Proficiency:**  This is fundamental for a Frontend Developer.  I'll expect to see evidence of strong JavaScript skills, including experience with modern frameworks and libraries (see below).\n*   **Years of Experience:** The job requires 5+ years. I will look for candidates who meet this requirement.\n\n**Medium Priority (Important, but not deal-breakers if slightly weaker):**\n\n*   **TensorFlow Experience:** This is a required skill. I'll be looking for specific projects or roles where the candidate used TensorFlow extensively. The more detail, the better.\n*   **Frontend Frameworks/Libraries:** While not explicitly listed, a strong Frontend Developer will likely have experience with frameworks like React, Angular, or Vue.js.  Experience with state management libraries (Redux, Vuex, etc.) and testing frameworks (Jest, Mocha, Cypress) is also a plus.\n*   **Collaboration/Teamwork:** The job description mentions collaborating with cross-functional teams. I'll look for evidence of teamwork, communication, and collaboration skills in their work experience descriptions.\n*   **Code Reviews/Testing:** The job description mentions conducting code reviews and testing. I'll look for evidence of experience with code reviews, unit testing, integration testing, and end-to-end testing.\n\n**Low Priority (Nice to Have, but not essential):**\n\n*   **Traveling (Hobbies):** This is a personality fit preference.  It won't significantly impact the rating unless the candidate explicitly mentions relevant skills gained through travel (e.g., adaptability, communication with diverse cultures).\n*   **Specific Company Names:** The company name is irrelevant to the candidate's skills.\n*   **Location Preference:** The job is in New Todd, El Salvador. The candidate's location is irrelevant as long as they are willing to relocate or work remotely (if the position allows).\n*   **Innovative Solutions:** This is a general statement. I'll be looking for specific examples of innovation in their work experience.\n\n**How I'll Assign Ratings (General Guidelines):**\n\n*   **9-10:**  Excellent match.  The candidate meets all required skills, has significant experience with the core technologies (Node.js, Docker, CI/CD, TensorFlow, JavaScript), and demonstrates strong teamwork and problem-solving abilities.  Their experience aligns well with the job description's responsibilities.\n*   **7-8:**  Good match.  The candidate meets most of the required skills and has some experience with the core technologies.  There might be a slight gap in one area, but overall, they are a strong contender.\n*   **5-6:**  Potentially suitable.  The candidate meets some of the required skills, but there are significant gaps in their experience.  They might require additional training or development to be fully effective in the role.\n*   **3-4:**  Weak match.  The candidate lacks several of the required skills and has limited experience with the core technologies.\n*   **1-2:**  Not a match.  The candidate's skills and experience are not relevant to the job description.\n\n**To get the best results, please provide the candidate resumes!**"

: 